<a href="https://colab.research.google.com/github/carlos-osorio/monitor-portuario/blob/main/notebooks/05_piso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd, glob

URL = ("https://raw.githubusercontent.com/carlos-osorio/monitor-portuario/"
       "main/data/portwatch_2026-07-15.csv")
df = pd.read_csv(URL, parse_dates=["date"])

PUERTOS = ["Buenaventura", "Cartagena", "Barranquilla", "Santa Marta"]
PERCENTIL = 0.05

for indicador in ["import", "export"]:
    print(f"\n===== {indicador.upper()} — regla de piso (P{int(PERCENTIL*100)}) =====")
    total = 0
    for puerto in PUERTOS:
        s = (df[df["portname"] == puerto].set_index("date")[indicador]
             .resample("W").sum().iloc[:-1])
        # piso = percentil 5 de las 52 semanas ANTERIORES (solo pasado, como el z)
        piso = s.rolling(52, min_periods=26).quantile(PERCENTIL).shift(1)
        alertas = s[(s < piso)]
        total += len(alertas)
        print(f"  {puerto:<14} {len(alertas):>3} alertas en {len(s)} semanas "
              f"({100*len(alertas)/len(s):.1f}%)")
    print(f"  → {total} alertas totales ≈ {total/7.5:.1f} por año (los 4 puertos, {indicador})")


===== IMPORT — regla de piso (P5) =====
  Buenaventura    19 alertas en 391 semanas (4.9%)
  Cartagena       20 alertas en 391 semanas (5.1%)
  Barranquilla    19 alertas en 391 semanas (4.9%)
  Santa Marta     25 alertas en 391 semanas (6.4%)
  → 83 alertas totales ≈ 11.1 por año (los 4 puertos, import)

===== EXPORT — regla de piso (P5) =====
  Buenaventura    29 alertas en 391 semanas (7.4%)
  Cartagena       25 alertas en 391 semanas (6.4%)
  Barranquilla    22 alertas en 391 semanas (5.6%)
  Santa Marta     21 alertas en 391 semanas (5.4%)
  → 97 alertas totales ≈ 12.9 por año (los 4 puertos, export)


In [2]:
import pandas as pd, numpy as np, glob

URL = ("https://raw.githubusercontent.com/carlos-osorio/monitor-portuario/"
       "main/data/portwatch_2026-07-15.csv")
df = pd.read_csv(URL, parse_dates=["date"])
PUERTOS = ["Buenaventura", "Cartagena", "Barranquilla", "Santa Marta"]

def z_mod(s, k=13):
    med = s.rolling(k).median().shift(1)
    mad = s.rolling(k).apply(lambda v: np.median(np.abs(v-np.median(v))), raw=True).shift(1)
    return 0.6745*(s-med)/mad

for indicador in ["import", "export"]:
    for P in [0.01, 0.02, 0.03]:
        total, ciegas = 0, 0
        fechas_bq = []
        for puerto in PUERTOS:
            s = (df[df["portname"]==puerto].set_index("date")[indicador]
                 .resample("W").sum().iloc[:-1])
            piso = s.rolling(52, min_periods=26).quantile(P).shift(1)
            z = z_mod(s)
            bajo_piso = s < piso
            total += bajo_piso.sum()
            # ¿cuántas de esas NO las habría cazado el z (z > -3)?
            ciegas += (bajo_piso & (z > -3)).sum()
            if puerto == "Barranquilla":
                fechas_bq = s[bajo_piso].index.strftime("%Y-%m-%d").tolist()
        print(f"{indicador} P{int(P*100)}: {total} totales ({total/7.5:.1f}/año), "
              f"{ciegas} que el z NO ve. Barranquilla: {fechas_bq}")

import P1: 34 totales (4.5/año), 32 que el z NO ve. Barranquilla: ['2020-02-16', '2020-07-19', '2020-07-26', '2021-08-22', '2021-10-03', '2021-11-28', '2024-08-04', '2024-11-17']
import P2: 53 totales (7.1/año), 50 que el z NO ve. Barranquilla: ['2019-07-28', '2020-02-16', '2020-07-19', '2020-07-26', '2021-07-25', '2021-08-22', '2021-10-03', '2021-11-28', '2022-08-07', '2024-08-04', '2024-11-17', '2025-09-14']
import P3: 63 totales (8.4/año), 60 que el z NO ve. Barranquilla: ['2019-07-28', '2020-02-16', '2020-07-19', '2020-07-26', '2021-07-25', '2021-08-22', '2021-10-03', '2021-11-28', '2022-08-07', '2024-08-04', '2024-11-17', '2024-12-22', '2025-09-14']
export P1: 45 totales (6.0/año), 43 que el z NO ve. Barranquilla: ['2019-07-21', '2019-09-08', '2020-11-22', '2022-09-18', '2023-02-19', '2023-04-23', '2023-07-02', '2025-01-05', '2025-02-09', '2025-06-01', '2025-08-17', '2025-09-28', '2026-07-05']
export P2: 64 totales (8.5/año), 60 que el z NO ve. Barranquilla: ['2019-07-21', '2019-0